# 03 — Explore the power market

The research uses three complete hourly SMARD CSV exports for local calendar year 2025. Computation takes place in Microsoft Fabric against `lh_power_renewables`. Notebook boundaries are persisted data contracts rather than shared Python session variables.

Original Fabric execution results are preserved separately in `evidence`. The current execution status of each refactored notebook is recorded in `notebook_status.json`.

Inputs are persisted silver tables. Exploratory calculations are read-only and do not publish analytical gold tables.

Current question: does consumption coincide with expensive hours? Compare time-average price with load-weighted price. Weights are hourly grid-load MWh, not monthly totals. Hourly price times hourly volume is an exposure estimate, not actual bills or exact sub-hourly settlement cost.

Next investigations, not yet implemented: wind/solar versus price within comparable months and hours; generation-weighted prices; negative-price exposure. Association alone does not establish causation.

In [ ]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt

spark.conf.set("spark.sql.session.timeZone", "UTC")

for table_name in (
    "silver.consumption_hour", "silver.generation_hour",
    "silver.price_hour", "silver.dim_time_hour",
):
    print(f"{table_name}: {spark.table(table_name).count()} rows")

## Check join coverage and reconciliation

In [ ]:
from decimal import Decimal

cross_source_checks = spark.sql("""
    WITH wind_solar AS (
        SELECT
            time_key,
            country_code,
            SUM(generation_mwh) AS wind_solar_mwh
        FROM silver.generation_hour
        WHERE generation_source_code IN (
            'wind_offshore',
            'wind_onshore',
            'solar'
        )
        GROUP BY time_key, country_code
    )
    SELECT
        COUNT(*) AS row_count,
        COUNT(p.time_key) AS matched_price_hours,
        COUNT(g.time_key) AS matched_generation_hours,
        MAX(
            ABS(
                c.grid_load_mwh
                - g.wind_solar_mwh
                - c.residual_load_mwh
            )
        ) AS max_residual_error_mwh
    FROM silver.consumption_hour AS c
    LEFT JOIN silver.price_hour AS p
        ON c.time_key = p.time_key
        AND p.bidding_zone_code = 'DE-LU'
    LEFT JOIN wind_solar AS g
        ON c.time_key = g.time_key
        AND c.country_code = g.country_code
    WHERE c.country_code = 'DE'
""")

display(cross_source_checks.toPandas())

result = cross_source_checks.first().asDict()

assert result["row_count"] == 8760, (
    "Unexpected joined row count"
)
assert result["matched_price_hours"] == 8760, (
    "Missing matching price hours"
)
assert result["matched_generation_hours"] == 8760, (
    "Missing matching generation hours"
)
assert result["max_residual_error_mwh"] <= Decimal("0.03"), (
    "Residual load reconciliation exceeds rounding tolerance"
)

print("Cross-source checks passed")

## Monthly consumption-weighted price

In [ ]:
spark.table("silver.dim_time_hour").createOrReplaceTempView("time_hour")

monthly_price_profile = spark.sql("""
    SELECT
        DATE_FORMAT(t.local_date, 'yyyy-MM') AS month,
        COUNT(*) AS matched_hours,
        ROUND(SUM(c.grid_load_mwh) / 1000000, 3)
            AS consumption_twh,
        ROUND(AVG(p.day_ahead_price_eur_mwh), 2)
            AS average_price_eur_mwh,
        ROUND(
            SUM(
                p.day_ahead_price_eur_mwh * c.grid_load_mwh
            ) / SUM(c.grid_load_mwh),
            2
        ) AS load_weighted_price_eur_mwh
    FROM silver.consumption_hour AS c
    INNER JOIN silver.price_hour AS p
        ON c.time_key = p.time_key
    INNER JOIN time_hour AS t
        ON c.time_key = t.time_key
    WHERE c.country_code = 'DE'
        AND p.bidding_zone_code = 'DE-LU'
    GROUP BY DATE_FORMAT(t.local_date, 'yyyy-MM')
    ORDER BY month
""")

display(monthly_price_profile.toPandas())

In [ ]:
monthly_checks = monthly_price_profile.collect()
assert len(monthly_checks) == 12, "Unexpected month count"
assert sum(row["matched_hours"] for row in monthly_checks) == 8760, "Unexpected annual coverage"
hours_by_month = {row["month"]: row["matched_hours"] for row in monthly_checks}
assert hours_by_month["2025-03"] == 743, "Unexpected spring DST coverage"
assert hours_by_month["2025-10"] == 745, "Unexpected autumn DST coverage"

In [ ]:
import matplotlib.pyplot as plt

monthly = monthly_price_profile.orderBy("month").toPandas()

price_columns = [
    "average_price_eur_mwh",
    "load_weighted_price_eur_mwh",
]

for column in price_columns:
    monthly[column] = monthly[column].astype(float)

monthly["price_gap_eur_mwh"] = (
    monthly["load_weighted_price_eur_mwh"]
    - monthly["average_price_eur_mwh"]
)

fig, axes = plt.subplots(
    2, 1,
    figsize=(12, 8),
    sharex=True,
    constrained_layout=True,
)

axes[0].plot(
    monthly["month"],
    monthly["average_price_eur_mwh"],
    marker="o",
    label="Time-average price",
    color="#2563EB",
)

axes[0].plot(
    monthly["month"],
    monthly["load_weighted_price_eur_mwh"],
    marker="o",
    label="Load-weighted price",
    color="#D97706",
)

axes[0].set_title("Germany: monthly day-ahead prices, 2025")
axes[0].set_ylabel("EUR/MWh")
axes[0].legend(frameon=False)
axes[0].grid(axis="y", alpha=0.25)

axes[1].bar(
    monthly["month"],
    monthly["price_gap_eur_mwh"],
    color="#64748B",
)

axes[1].axhline(0, color="#111827", linewidth=0.8)
axes[1].set_title("Load-weighted price minus time-average price")
axes[1].set_ylabel("EUR/MWh")
axes[1].set_xlabel("Month")
axes[1].grid(axis="y", alpha=0.25)
axes[1].tick_params(axis="x", rotation=45)

plt.show()

## Interpretation of the original exported results
The original run showed a January gap of +4.35 EUR/MWh and a June gap of -0.39 EUR/MWh. These are observations from the preserved source export, not outputs of a new run of this refactored notebook. A positive gap indicates positive within-month association between price and load. A near-zero gap can reflect offsetting patterns and does not prove constant consumption.

For portable table outputs, the 12-row result is displayed as pandas rather than a Fabric-only widget. Matplotlib produces a standard static image. Notebook output is a snapshot; rerun after changing inputs.